<a href="https://colab.research.google.com/github/Taheri1400/deep-learning-projects/blob/main/title_categorizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 dir=rtl align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
دسته‌یاب عنوان
</font>
</h1>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
در آخرین مسئله‌ی پروژه‌ای باز به سراغ داده‌های وب‌سایت ترب خواهیم رفت. در پروژه‌ی دوم درباره‌ی اهمیت استخراج خودکار اطلاعات صحبت کردیم و مدلی ساختیم که بتواند با دریافت تصویر یک محصول، دسته‌بندی آن را پیش‌بینی کند. اما چرا از داده‌های متنی مرتبط با یک محصول همچون عنوان آن برای تشخیص دسته‌بندی استفاده نکنیم؟ با استفاده از چنین مدلی حتی اگر تصویری برای محصول ثبت نشده باشد نیز قادر به دسته‌بندی آن هستیم. علاوه‌براین می‌توانیم با استفاده از خروجی دو مدل تصویری و متنی با همدیگر، از پیش‌بینی خود مطمئن‌تر شویم. یعنی اگر هر دوی آن‌ها برای یک محصول یک دسته‌بندی مشابه را پیش‌بینی کنند، قطعیت خروجی کار ما بیشتر خواهد بود. پس در این مسئله با استفاده از تکنیک‌هایی که در درسنامه‌های این فصل آموختیم به ساخت یک مدل پردازش زبان طبیعی برای دسته‌بندی محصولات خواهیم پرداخت.
</font>
</p>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مجموعه‌داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3> فایل <code>torob_train.csv</code> شامل اطلاعاتی درباره‌ی تعدادی از محصولات لیست‌شده در این وب‌سایت بوده و شرح هرکدام از ستون‌های آن در جدول زیر آمده است. در صورت تمایل می‌توانید بخشی از داده‌های این مجموعه را به‌عنوان مجموعه‌ی اعتبارسنجی جدا کنید. فایل <code>torob_test.csv</code> نیز مربوط به نمونه‌های آزمون بوده که طبیعتاً شامل ستون متغیر هدف یعنی <code>cat_id</code> نیست و هدف این است که مدل شما بتواند دسته‌بندی محصول را پیش‌بینی کند.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>

| <b>نام ستون</b> | <b>توضیحات</b> |
| :---: | :---: |
| <code>name1</code> | عنوان اصلی محصول (معمولاً فارسی) |
| <code>name2</code> | عنوان دوم/اختیاری محصول (معمولاً انگلیسی) |
| <code>cat_id</code> | شناسه‌ی دسته‌ی اصلی محصول |

</font>
</div>
</center>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
دسته‌بندی‌های این محصولات و شناسه‌ی هر کدام در جدول زیر آمده است:‌
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>

| <b>شناسه‌ی دسته</b> | <b>عنوان دسته</b> |
| :---: | :---: |
| <code>0</code> | کاپشن، بارانی و پالتو مردانه |
| <code>1</code> | سویشرت و هودی مردانه |
| <code>2</code> | ساعت مچی عقربه‌ ای و دیجیتالی |
| <code>3</code> | ساعت دیواری، رومیزی و تزیینی |
| <code>4</code> | لوازم جانبی ساعت معمولی و هوشمند |
| <code>5</code> | سویشرت و هودی خردسال و نوجوان |
| <code>6</code> | کاپشن و پالتو خردسال و نوجوان |
| <code>7</code> | سویشرت ورزشی مردانه |
| <code>8</code> | سویشرت و شلوار ورزشی مردانه |
| <code>9</code> | ساک و چرخ خرید |
| <code>10</code> | چمدان و ساک |

</font>
</div>
</center>

In [1]:
import pandas as pd
import numpy as np

In [2]:
torob_train=pd.read_csv('torob_train.csv')
torob_test=pd.read_csv('torob_test.csv')   # Reading/Loading the dataset files
# TODO

In [8]:
torob_train.isna().sum()

,0
name1,0
name2,8226
cat_id,0


In [9]:
torob_train['name2'] = torob_train['name2'].fillna('')

In [14]:
torob_test['name2'] = torob_test['name2'].fillna('')


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پیش‌پردازش
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
پیش‌پردازش متن یکی از تاثیرگذارترین عوامل در عملکرد مدل است زیرا کلمات موجود در متن‌ها محدود به یک لغت‌نامه‌ی خاص نیستند و افراد می‌توانند یک واژه را به‌شکل‌های مختلفی دگرگون کنند. بنابراین ما برای آن‌که کار مدل را راحت‌تر کنیم و از پیچیدگی محاسباتی بکاهیم سعی می‌کنیم متن‌ها را نرمال کنیم. به این معنی که مثلاً کلمات/توکن‌های نادر یا بی‌اهمیت را از آن‌ها حذف کنیم یا واژه‌هایی با یک معنی مشابه را تنها با یک فرم از آن‌ها جایگزین کنیم.
</font>
</p>

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تکنیک‌های پیش‌پردازش
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
 تکنیک‌های پیش‌پردازش و پاک‌سازی متفاوتی برای متن وجود دارد که هم می‌توانید با جست‌وجو در اینترنت با آن‌ها آشنا شوید و هم می‌توانید از ایده‌های خلاقانه خودتان استفاده کنید. اما در لیست زیر به چند نمونه از رایج‌ترین آن‌ها اشاره شده است. توجه داشته باشید انتخاب این‌که کدام پیش‌پردازش مناسب این مسئله است بر عهده‌ی خودتان است و نیازمند این است که درباره‌ی تاثیرشان بر روی هدف مسئله تفکر کنید.
</font>
</p>
<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<li><b>حذف کلمات توقف (stop words):</b> برخی کلمات هستند که در متن‌ها به تعداد بسیار زیادی تکرار می‌شوند اما بار معنایی خاصی ندارند و معمولاً در مسائل مرتبط با پردازش زبان نه تنها تاثیر مثبتی در عملکرد مدل نمی‌گذارند بلکه پرتکرار بودن آن‌ها ارزش زیادی را به این کلمات می‌دهد. از جمله‌ی این کلمات در فارسی می‌توان به «از»، «به»، «است»، «آنان»، «آخر»، «آنجا» و غیره اشاره کرد. شما می‌توانید در متن خود با توجه به مسئله‌ای که تعریف می‌کنید نسبت به حذف این کلمات اقدام کنید. البته توجه داشته باشید که برخی کلمات می‌‍توانند به‌صورت رایج کلمه‌ی توقف شناخته شوند اما در مسئله‌ی خاص شما اتفاقاً با اهمیت باشند.</li>
<li><b>حذف اعداد و حروف اضافه: </b>اعداد و حروف اضافه نیز همچون کلمات توقف ممکن است با توجه به مسئله بی‌اهمیت باشند و بتوان آن‌ها را به‌کلی حذف کرد. البته ایده‌ی دیگری وجود دارد که به‌جای هر عدد یک توکن خاص مثلاً <code>NUMBER</code> قرار دهید.</li>
<li><b>حذف شماره تماس/ایمیل/لینک یا تگ‌های <code>html</code>: </b>از آن‌جا که تگ‌های <code>html</code> یا آدرس ایمیل/لینک/شماره‌تماس مقادیر محدودی ندارند و اطلاعات زیادی را برای مدل فراهم نمی‌کنند معمولاً در صورت وجود این جنس داده در متن‌های خود، آن‌ها را از متن‌ها حذف می‌کنیم. البته می‌توان به‌جای حذف، آن‌ها را با یک توکن خاص همچون <code>EMAIL</code> یا <code>PHONE</code> جایگزین کرد.</li>
<li><b>نرمال‌سازی (normalization):</b> انجام برخی اصلاحات رایج در متن جهت نرمال کردن کلمات نیز تکنیک دیگری است که کاربرد زیادی دارد. به عنوان مثال در زبان فارسی رعایت نیم‌فاصله می‌تواند یک نرمال‌سازی بسیار مفید شناخته شود. اگر این کار صورت نگیرد به عنوان مثال کلمه‌ی «نیم فاصله» با کلمه‌ی «نیم‌فاصله» کاملاً متفاوت در نظر گرفته می‌شود. اولی خود شامل دو کلمه‌ی مجزاست در حالی‌که دومی یک کلمه‌ی واحد است.</li>
<li><b>ریشه‌یابی (stemming):</b> تکنیک دیگر تبدیل کلمات به ریشه‌ی آن‌هاست. به عنوان مثال در زبان فارسی «کتاب‌ها» را می‌توان با ریشه‌ی خود یعنی «کتاب» جایگزین کنیم. با این جایگزینی مفهوم کلمه عوض نخواهد شد اما کار ماشین با متن بسیار راحت می‌شود زیرا در این حالت بین این دو تمایزی قائل نمی‌شود.</li>
</ul>
</font>
</p>

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
ابزارهای پیش‌پردازش
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
برای پیش‌پردازش متون فارسی کتابخانه‌ها و ابزارهای مختلف (اما نه‌چندان جامع و حرفه‌ای) وجود دارد که می‌توانید به دلخواه خود از آن‌ها استفاده کنید. البته بسیاری از پیش‌پردازش‌ها را می‌توانید به‌صورت دستی و با عبارات منظم (<code>regex</code>) نیز پیاده‌سازی کنید. با این حال برخی از ابزارهایی که می‌تواند به کمک‌تان بیاید در ادامه لیست شده است.
</font>
</p>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<li>یکی از کتابخانه‌های بسیار مشهور پردازش زبان طبیعی، <code>nltk</code> است که امکانات مختلفی را در دسترس‌مان قرار می‌دهد. اگر می‌خواهید پیش‌پردازش‌هایی را بر روی متن اعمال کنید که محدود به زبان فارسی نیست می‌توانید از این کتابخانه بهره ببرید. به‌عنوان مثال اگر نیاز به توکن‌سازی (تبدیل متن به لیست توکن) دارید یا می‌خواهید حروف اضافه/علائم نگارشی یک متن را بیابید این کتابخانه گزینه‌ی خوبی‌ست. برای مشاهده‌ی مستندات آن می‌توانید به <a href="https://www.nltk.org/" target=”_blank”>این لینک</a> مراجعه کنید.</li>
<li>در درسنامه‌ی این فصل با کتابخانه‌ی <code>DadmaTools</code> (<a href="https://github.com/Dadmatech/DadmaTools/tree/main" target=”_blank”>لینک مخزن</a>) آشنا شدید. این کتابخانه با آن‌که نسبتاً جدید است و در حال حاضر مستندات کافی هم ندارد اما امکانات حرفه‌ای و متنوعی را در اختیارتان قرار می‌دهد. به‌عنوان مثال با کلاس <code>Normalizer</code> آن می‌توانید متن‌های خود را از جنبه‌های مختلفی نرمال کنید. علاوه‌براین، همان‌طور که در درسنامه دیدید، این کتابخانه بردارهای تعبیه‌ی پیش‌آموخته برای زبان فارسی را نیز در اختیارتان قرار می‌دهد. البته توجه داشته باشید که پکیج و فایل‌های آن از نظر حجمی سنگین است و ترجیحاً از آن در محیط کولب استفاده کنید تا نیاز به دانلود فایل‌های سنگین نداشته باشید. همچنین در حال حاضر این کتابخانه با کراس ۳ سازگار نیست و پیشنهاد می‌کنیم ابتدا به‌صورت جداگانه داده‌های خود را با آن پاک‌سازی و ذخیره کنید، سپس به‌سراغ مدل‌سازی با کراس بروید.</li>
<li>کتابخانه‌ی <a href="https://www.roshan-ai.ir/hazm/" target=”_blank”>هضم</a> امکانات مختلفی نظیر نرمال‌سازی یا ریشه‌یابی را در اختیارتان قرار می‌دهد. با این‌که عملکرد آن بسیار دقیق نیست اما اگر در نصب آن با مشکلی مواجه نمی‌شوید انتخاب مناسبی‌ست.</li>
<li>یک انتخاب حرفه‌ای‌تر که الگوریتم‌ها و مدل‌های آن بر پایه‌ی پژوهش‌های علمی زبان‌های مختلف توسعه یافته‌اند، کتابخانه‌ی پردازش زبان طبعی استنفورد با نام <a href="https://www.roshan-ai.ir/hazm/" target=”_blank”>Stanza</a> است که از زبان فارسی نیز پشتیبانی می‌کند.</li>
<li>همچنین لیستی از کلمات توقف فارسی را می‌توانید از <a href="https://github.com/JoyeBright/stopwords_guilannlp" target=”_blank”>این لینک/پکیج</a> بیابید.</li>

</ul>
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
<span style="color:red"><b>توجه:</b></span>
پیشنهاد می‌کنیم پس از پیش‌پردازش متن‌ها، حاصل کار را در فایل‌های جدید <code>csv</code> ذخیره کنید تا در صورت ریست شدن محیط یا مشکلات دیگر (نظیر تداخل نسخه‌ی کتابخانه‌های پیش‌پردازش با کراس ۳) نیازی به اجرای مجدد مراحل پیش‌پردازش و اتلاف وقت نباشد.
</font>
</p>

In [17]:
import re
import pandas as pd

def preprocess_text(text):
    text = str(text)

    # تبدیل حروف عربی به شکل رایج فارسی
    text = text.replace('ي', 'ی')
    text = text.replace('ك', 'ک')

    # حذف لینک‌ها و ایمیل‌ها
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)

    # یکسان‌سازی فاصله‌ها
    text = re.sub(r'\s+', ' ', text)

    # حذف فاصله‌های ابتدا و انتهای متن
    text = text.strip()

    return text


# جایگزینی مقادیر خالی عنوان دوم
torob_train['name2'] = torob_train['name2'].fillna('')
torob_test['name2'] = torob_test['name2'].fillna('')

# ترکیب عنوان فارسی و انگلیسی
torob_train['text'] = (
    torob_train['name1'] + ' ' + torob_train['name2']
)

torob_test['text'] = (
    torob_test['name1'] + ' ' + torob_test['name2']
)

# اعمال پیش‌پردازش
torob_train['text'] = torob_train['text'].apply(preprocess_text)
torob_test['text'] = torob_test['text'].apply(preprocess_text)

# مشاهده چند نمونه
print(torob_train[['text', 'cat_id']].head())

                                                text  cat_id
0                   کاپشن کوهنوردی پر سنگین نورث فیس       0
1                       پالتو خزدار مردانه مارک اصلی       0
2                              کاپشن مردانه مدل Bako       0
3  کاپشن سالامون salomon پارچه گورتکس کره ای اعلا...       0
4                  کاپشن‌‌‌ نظامی‌‌‌ آمریکایی‌‌‌ سبز       0


In [18]:
X_train = torob_train['text']
y_train = torob_train['cat_id']

X_test = torob_test['text']

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مدل‌سازی
</font>
</h2>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
همچون پروژه‌های دیگر در اینجا نیز دست‌تان باز است که هر الگوریتم یا ساختاری را آزمایش کنید.
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<li>می‌توانید ابتدا عملکرد یک مدل یادگیری ماشین ساده مثل رگرسیون لجستیک را به‌عنوان مدل پایه (baseline) ارزیابی کنید و با استفاده از آن تاثیر پیش‌پردازش‌های مختلف روی داده‌ها را نیز بسنجید،‌ سپس به‌سراغ شبکه‌های عصبی بروید.</li>
<li>در شبکه‌ی عصبی نیز می‌توانید ساختارهای متنوعی را آزمایش کنید. به‌عنوان مثال می‌توانید یک شبکه‌ی ساده شامل لایه‌های <code>Dense</code> پیاده کنید که بردار <code>tf-idf</code> هر متن را ورودی می‌گیرد.</li>
<li>یا می‌توانید از لایه‌ی تعبیه‌ی کراس (<code>Embedding</code>) استفاده کرده و آن را به یک مدل دنباله‌ای ورودی دهید. به‌عنوان راهنمایی توجه داشته باشید در صورت استفاده از مدل دنباله‌ای آرگومان <code>return_sequences</code> آن را فعال کرده باشید و خروجی را به لایه‌ای همچون <code>GlobalMaxPool1D</code> دهید (درباره‌ی این ساختار بیشتر فکر کنید). </li>
<li>همچنین می‌توانید مشابه با کدی که در داخل درسنامه نوشته شده، از بردارهای تعبیه‌ی پیش‌آموخته استفاده کنید. می‌توانید هر دو حالت فریز کردن لایه‌ی تعبیه یا آموزش مجدد آن را آزمایش کنید.</li>
<li>حتی می‌توانید خفن‌تر (😎) عمل کرده و به جست‌وجو درباره‌ی معماری‌های دیگر بپردازید. به‌عنوان مثال حتی می‌توانید از یک شبکه‌ی عصبی کانولوشنی بهره ببرید و درباره‌ی این‌که چرا این نوع شبکه می‌توانید برای چنین مسئله‌ای پاسخ خوبی تولید کند بیشتر تفکر کنید.</li>
</ul>
</font>
</p>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="orange"><b color='orange'>نکته:</b></font>
<font face="vazir" size=3>
در مسائل واقعی لزوماً همواره مدل یادگیری عمیق نسبت به الگوریتم‌های سنتی یادگیری ماشین عملکرد بهتری نخواهد داشت یا ممکن است اختلاف عملکرد آن فاحش نباشد. این موضوع کاملاً به نوع و میزان داده‌های شما، هدف و دامنه‌ی مسئله و بسیاری از عوامل دیگر بستگی دارد. با این حال طبق نتایجی که از آزمایش‌های خود در این پروژه به دست می‌آورید سعی کنید بیشتر تفکر کرده و تفسیری برای آن‌ها ارائه دهید.
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
<span style="color:purple"><b>🧐 بیشتر بی‌اندیشیم:</b></span>
پیشنهاد می‌کنیم حاصل تفکرات خود درباره‌ی سوالات مطرح‌شده در بالا و تفسیرهای خود را در صفحه‌ی تمرین نوشته و دیدگاه خود را با دیگران به‌اشتراک بگذارید. علاوه‌براین پیشنهاد می‌کنیم هر روش خلاقانه‌ی دیگری که به ذهن‌تان می‌رسد را نیز آزمایش کنید.
</font>
</p>

In [20]:
import numpy as np
import keras
from keras import layers
from keras.src.legacy.preprocessing.text import Tokenizer
#from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences

MAX_WORDS = 20000
MAX_LEN = 40

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')

tokenizer.fit_on_texts(X_train)

x_train_seq = tokenizer.texts_to_sequences(X_train)
x_test_seq = tokenizer.texts_to_sequences(X_test)

x_train_pad = pad_sequences(
    x_train_seq, maxlen=MAX_LEN, padding='post', truncating='post'
)

x_test_pad = pad_sequences(
    x_test_seq, maxlen=MAX_LEN, padding='post', truncating='post'
)

In [21]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),

    layers.Embedding(
        input_dim=MAX_WORDS,
        output_dim=64
    ),

    layers.GlobalMaxPooling1D(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(11, activation='softmax')
])

In [22]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 64)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,284,875 (4.90 MB)

 Trainable params: 1,284,875 (4.90 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = model.fit(
    x_train_pad,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.3295 - loss: 2.1916 - val_accuracy: 0.0153 - val_loss: 3.5330
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.6678 - loss: 1.2949 - val_accuracy: 0.0918 - val_loss: 4.6226
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8471 - loss: 0.5863 - val_accuracy: 0.0952 - val_loss: 5.1640
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9068 - loss: 0.3453 - val_accuracy: 0.0963 - val_loss: 5.3997
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9389 - loss: 0.2398 - val_accuracy: 0.0963 - val_loss: 5.5453
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9555 - loss: 0.1777 - val_accuracy: 0.0963 - val_loss: 5.7901
Epoch 7/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9691 - loss: 0.1351 - val_accuracy: 0.0963 - val_loss: 6.0031
Epoch 8/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9758 - loss: 0.1089 - val_accuracy: 0.0963 - v

In [28]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# استفاده از متن‌های آماده‌شده
X = torob_train['text'].fillna('')
y = torob_train['cat_id']

# تقسیم داده‌ها برای ارزیابی
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# تبدیل متن به ویژگی‌های عددی
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    min_df=2,
    max_features=300000,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

# ساخت مدل
model = LogisticRegression(
    C=4,
    max_iter=2000,
    solver='lbfgs'
)

# آموزش
model.fit(X_train_tfidf, y_train)

# ارزیابی
y_val_pred = model.predict(X_val_tfidf)

accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {accuracy * 100:.2f}%')

Validation Accuracy: 95.86%


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پیش‌بینی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
در فایل <code>torob_test.csv</code> لیستی از نمونه‌های آزمون در اختیار شما قرار گرفته و نیاز است مدل شما تا حد ممکن دسته‌بندی هر کدام از آن‌ها را به‌درستی تشخیص دهد. پیشنهاد می‌کنیم برای اطمینان از عملکرد مدل خود و بهبود آن، ابتدا آن را بر روی یک مجموعه‌ی اعتبارسنجی آزمایش کنید.
</font>
</p>


<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معیار ارزیابی
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
هدف این است که مدل شما دسته‌بندی یک محصول را با دقت بالا پیش‌بینی کند، بنابراین در این پروژه از معیار  <code>Accuracy</code> جهت ارزیابی نتایج مدل شما استفاده خواهد شد.
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="red"><b color='red'>توجه:</b></font>
<font face="vazir" size=3>
 جهت کسب امتیاز کامل نیاز است تا پاسخ شما حداقل امتیاز <code>95.30</code> را با توجه به معیار معرفی‌شده کسب نماید.
</font>
</p>

In [24]:
val_accuracy = max(history.history['val_accuracy'])
print('Best Validation Accuracy:', val_accuracy)

Best Validation Accuracy: 0.09631728380918503


<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
ساختار خروجی برای نمونه‌های آزمون
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
 پیش‌بینی مدل خود را برای نمونه‌های آزمون به‌شکل برچسبی بین ۰ تا ۱۰ تولید کرده و در ستون <code>cat_id</code> یک دیتافریم با نام <code>submission</code> در قالب جدول زیر ذخیره کنید. توجه داشته باشید ترتیب پیش‌بینی‌های شما در این ستون باید دقیقاً مشابه با نمونه‌های آزمون باشد. یعنی ردیف اول بیانگر پیش‌بینی نمونه‌ی آزمون اول باشد.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|<code>cat_id</code>|برچسب/شناسه‌ی دسته (پیش‌بینی مدل)|
    
</font>
</div>
</center>


In [37]:
# تبدیل متن‌های آزمون به ویژگی‌های TF-IDF
X_test = torob_test['text'].fillna('')
X_test_tfidf = vectorizer.transform(X_test)

# پیش‌بینی دسته‌بندی
y_test_pred = model.predict(X_test_tfidf)

# ساخت دیتافریم خروجی
submission = pd.DataFrame({
    'cat_id': y_test_pred.astype(int)
})

# ذخیره فایل
submission.to_csv('submission.csv', index=False)

print(submission.head())
print(submission.shape)

   cat_id
0       9
1       7
2       3
3      10
4       3
(1201, 1)


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد. همچنین اگر از گوگل کولب استفاده می‌کنید، در صورت نیاز به پشتیبانی حتماً آخرین نسخه از نت‌بوک را به‌صورت دستی دانلود کرده و داخل فایل ارسالی قرار دهید یا لینک کولب را با ما به‌اشتراک بگذارید.
</font>
</p>

In [39]:
import zipfile
import joblib
import os

if not os.path.exists(os.path.join(os.getcwd(), 'title_categorizer.ipynb')):
    %notebook -e title_categorizer.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['title_categorizer.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['title_categorizer.ipynb', 'submission.csv']
